In [0]:
%run ./configuration_file

In [0]:
races_schema=StructType(fields=[StructField("raceId", IntegerType(), False),
                                 StructField("year", IntegerType(), True),
                                 StructField("round", IntegerType(), True),
                                 StructField("circuitId", IntegerType(), False),
                                 StructField("name", StringType(), True),
                                 StructField("date", DateType(), True),
                                 StructField("time", TimestampType(), True),
                                 StructField("url", StringType(), True)])


In [0]:
races=spark.read.option('header', True).schema(races_schema).csv(bronze+'/races.csv')

In [0]:
races= races.withColumnRenamed("circuitId", "circuit_id")\
            .withColumnRenamed("raceId", "race_id")\
            .withColumnRenamed('year', 'race_year')\
            .withColumn('ingestion_date', current_date())\
            .withColumn('race_timestamp', to_timestamp(concat(col('date'), lit(' '), date_format(col('time'), 'HH:mm:ss')), 'yyyy-MM-dd HH:mm:ss'))\
            .drop('url')\
            .drop("date")\
            .drop('time')

In [0]:
races.write.mode("overwrite").partitionBy("race_year").parquet(silver+'/races_tb')

In [0]:
df=spark.read.parquet(silver+'/races_tb')
display(df)